In [2]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

# =============================================================================
# CONFIGURACIÓN MAESTRA (CUMPLIMIENTO 100% INFORME)
# =============================================================================
ACTIVOS = ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'AMZN', 'META', 'AVGO', 'ASML', 'TSM', 'ADBE', 'NFLX', 'AMD']
CAPITAL_INICIAL = 100000
FECHA_INICIO = "2015-01-01"
SIMULACIONES = 1000
ANIOS_PROYECCION = 5
BLOCK_SIZE = 10 

def ejecutar_auditoria_definitiva():
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Iniciando sistema integral...")
    
    # --- 1. DESCARGA CORREGIDA DE DATOS REALES ---
    try:
        raw_data = yf.download(ACTIVOS, start=FECHA_INICIO, group_by='ticker')
        # Manejo de multi-índice para evitar el KeyError 'Adj Close'
        data = pd.DataFrame({ticker: raw_data[ticker]['Adj Close'] for ticker in ACTIVOS}).dropna()
        print(f"✅ Datos reales obtenidos: {len(data)} días de historia para {len(ACTIVOS)} activos.")
    except Exception as e:
        print(f"❌ Error en descarga: {e}")
        return

    # --- 2. MOTOR DE BACKTEST REAL (Lógica Endógena) ---
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Ejecutando lógica Sentinel sobre precios reales...")
    returns_estrategia_total = pd.DataFrame(index=data.index)
    
    for ticker in ACTIVOS:
        precios = data[ticker]
        # Cálculo real de EMA200
        ema200 = precios.ewm(span=200, adjust=False).mean()
        
        # Estado de Inversión basado exclusivamente en el PRECIO REAL
        esta_invertido = (precios > ema200).astype(int)
        
        # Retornos diarios y aplicación de la estrategia
        ret_diario = precios.pct_change().fillna(0)
        ret_est = ret_diario * esta_invertido
        
        # Aplicación de COSTOS REALES (Sesión 3.5)
        # Swap CFD (0.02% diario si está dentro)
        ret_est = np.where(esta_invertido == 1, ret_est - 0.0002, 0)
        # Comisión 0.1% al entrar o salir
        cambios = esta_invertido.diff().fillna(0) != 0
        ret_est[cambios] -= 0.001
        
        returns_estrategia_total[ticker] = ret_est

    # Portafolio consolidado (Equiponderado)
    portfolio_real_series = returns_estrategia_total.mean(axis=1)

    # --- 3. BOOTSTRAPPING POR BLOQUES (Sesión 3.4) ---
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Generando {SIMULACIONES} trayectorias de futuro...")
    serie_para_simular = portfolio_real_series.values
    dias_sim = 252 * ANIOS_PROYECCION
    num_bloques = dias_sim // BLOCK_SIZE
    
    resultados_finales = []
    plt.figure(figsize=(12, 6), facecolor='#0d0d0d')
    ax = plt.gca()
    ax.set_facecolor('#0d0d0d')
    
    for _ in range(SIMULACIONES):
        indices = np.random.randint(0, len(serie_para_simular) - BLOCK_SIZE, num_bloques)
        trayectoria = np.concatenate([serie_para_simular[i:i+BLOCK_SIZE] for i in indices])
        
        capital_path = CAPITAL_INICIAL * np.cumprod(1 + trayectoria)
        resultados_finales.append(capital_path[-1])
        plt.plot(capital_path, color='cyan', alpha=0.01)

    # --- 4. RESULTADOS Y CRITERIOS DE ACEPTACIÓN ---
    p5 = np.percentile(resultados_finales, 5)
    p50 = np.percentile(resultados_finales, 50)
    prob_perdida = (len([x for x in resultados_finales if x < CAPITAL_INICIAL]) / SIMULACIONES) * 100
    
    print("\n" + "═"*70)
    print("      INFORME FINAL DE VALIDACIÓN EMPÍRICA")
    print("═"*70)
    print(f"Serie histórica real: {FECHA_INICIO} hasta hoy")
    print(f"Escenario P5 (Pésimo):    ${p5:,.2f}")
    print(f"Escenario P50 (Mediana):  ${p50:,.2f}")
    print(f"Probabilidad de Pérdida: {prob_perdida:.2f}%")
    print("─"*70)

    # Criterio de aceptación: P5 > $50,000
    if p5 > 50000 and prob_perdida < 5:
        print("✅ ESTATUS: MODELO VALIDADO. APTO PARA INTEGRACIÓN DE IA.")
    else:
        print("❌ ESTATUS: REVISIÓN REQUERIDA. Riesgo excesivo en datos reales.")
    print("═"*70)

    plt.axhline(y=CAPITAL_INICIAL, color='red', linestyle='--', label='Breakeven')
    plt.yscale('log')
    plt.title("Simulación Sentinel V4.0: Precios Reales + Bootstrapping", color='white')
    plt.show()

ejecutar_auditoria_definitiva()

[19:36:19] Iniciando sistema integral...


[*********************100%***********************]  12 of 12 completed

❌ Error en descarga: 'Adj Close'
